# Gap Module — Supply Chain  ·  OWASP LLM03

The model, its weights, its training data, its dependencies and its plugins all arrive from somewhere.
A compromise anywhere in that chain — a backdoored model on a hub, a pickle that runs code on load, a
typosquatted package, an unpinned dependency — lands inside your trust boundary before the model ever
runs. This is OWASP LLM03, absent from the Udemy course.

Each cell demonstrates a chain risk and its control: integrity verification, safe deserialisation,
dependency pinning, typosquat detection, and an AI bill of materials. Local, no live targets.

In [1]:
import hashlib, json, os
print("supply-chain controls: verify before trust")

supply-chain controls: verify before trust


## 1. Model & Weight Integrity — verify the hash before loading

A model pulled from a hub can be swapped for a backdoored copy. The control: pin the expected hash of
the artifact and refuse to load anything that does not match.

In [2]:
# a "model file" we trust, and its known-good hash recorded at vetting time
trusted_blob = b"<<legit model weights v1.0>>"
KNOWN_GOOD_SHA256 = hashlib.sha256(trusted_blob).hexdigest()

def load_model(blob):
    digest = hashlib.sha256(blob).hexdigest()
    if digest != KNOWN_GOOD_SHA256:
        raise ValueError(f"INTEGRITY FAIL: {digest[:12]}... != pinned {KNOWN_GOOD_SHA256[:12]}...")
    return "model loaded"

print("legit artifact :", load_model(trusted_blob))
tampered = b"<<legit model weights v1.0>> + backdoor"
try: load_model(tampered)
except ValueError as e: print("tampered artifact:", e)

legit artifact : model loaded
tampered artifact: INTEGRITY FAIL: 0ae75934bd7b... != pinned a87c0f910195...


## 2. Malicious Model Backdoor — code execution on load (pickle)

Many model/checkpoint formats are Python pickles, and **unpickling runs code**. A poisoned checkpoint
executes the moment you load it — before any inference. The control: never unpickle untrusted data;
use a non-executable format (safetensors) or restrict the unpickler.

In [3]:
import pickle, io

class Evil:
    def __reduce__(self):
        # runs ON unpickle. print is a harmless picklable stand-in; a real
        # payload would run os.system / exec here, before any inference.
        return (print, ("    [!] ATTACKER CODE EXECUTED DURING UNPICKLE",))

poisoned = pickle.dumps(Evil())
print("loading the poisoned checkpoint (VULNERABLE):")
pickle.loads(poisoned)          # the attacker's print fires here — arbitrary code ran

# SAFE: a restricted unpickler that refuses to import arbitrary callables
class SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        raise pickle.UnpicklingError(f"blocked {module}.{name}")
print("\nloading with a restricted unpickler (SAFE):")
try:
    SafeUnpickler(io.BytesIO(poisoned)).load()
except pickle.UnpicklingError as e:
    print("   ", e, "<- refused before any code ran")
print("\nbest practice: ship weights as safetensors (data-only, cannot execute)")

loading the poisoned checkpoint (VULNERABLE):
    [!] ATTACKER CODE EXECUTED DURING UNPICKLE

loading with a restricted unpickler (SAFE):
    blocked builtins.print <- refused before any code ran

best practice: ship weights as safetensors (data-only, cannot execute)


## 3. Dependency Confusion & Pinning

An unpinned or internal-named dependency can be shadowed by a malicious public package of the same
name (dependency confusion). The control: pin exact versions **and** hashes in a lockfile.

In [4]:
# a requirements line with no version is a moving target an attacker can hijack
unpinned = ["requests", "langchain", "internal-rag-utils"]
pinned   = {"requests": "2.32.3", "langchain": "0.3.7", "internal-rag-utils": "1.2.0"}
LOCK_HASHES = {"requests": "sha256:abc123...", "langchain": "sha256:def456..."}

def audit(reqs):
    issues = []
    for r in reqs:
        if "==" not in r and r not in pinned:
            issues.append(f"UNPINNED: {r} (resolves to whatever the index serves today)")
        if r == "internal-rag-utils":
            issues.append(f"CONFUSION RISK: {r} is an internal name — a public package could shadow it")
    return issues

for i in audit(unpinned): print(" ", i)
print("\ncontrol: pin exact versions + hashes, host internal packages on a private index")
print("pinned+hashed:", {k: (pinned[k], LOCK_HASHES.get(k,'needs hash')) for k in pinned})

  CONFUSION RISK: internal-rag-utils is an internal name — a public package could shadow it

control: pin exact versions + hashes, host internal packages on a private index
pinned+hashed: {'requests': ('2.32.3', 'sha256:abc123...'), 'langchain': ('0.3.7', 'sha256:def456...'), 'internal-rag-utils': ('1.2.0', 'needs hash')}


## 4. Typosquatting / Slopsquatting Detection

Attackers publish packages whose names are one edit from a popular one — or match names LLMs
hallucinate ("slopsquatting"). The control: check install names against a trusted set by edit
distance.

In [5]:
POPULAR = {"numpy","pandas","requests","langchain","transformers","openai","scikit-learn","huggingface-hub"}

def edit_distance(a, b):
    dp = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        prev, dp[0] = dp[0], i
        for j, cb in enumerate(b, 1):
            prev, dp[j] = dp[j], min(dp[j]+1, dp[j-1]+1, prev+(ca!=cb))
    return dp[-1]

def typosquat_check(name):
    if name in POPULAR: return "ok (exact match)"
    near = [(p, edit_distance(name, p)) for p in POPULAR]
    p, d = min(near, key=lambda x: x[1])
    return f"SUSPICIOUS: 1-{d} edits from '{p}'" if d <= 2 else "unknown package — vet manually"

for pkg in ["numpy", "numpyy", "pandaa", "requsts", "transformerss", "my-internal-tool"]:
    print(f"  {pkg:18} -> {typosquat_check(pkg)}")

  numpy              -> ok (exact match)
  numpyy             -> SUSPICIOUS: 1-1 edits from 'numpy'
  pandaa             -> SUSPICIOUS: 1-1 edits from 'pandas'
  requsts            -> SUSPICIOUS: 1-1 edits from 'requests'
  transformerss      -> SUSPICIOUS: 1-1 edits from 'transformers'
  my-internal-tool   -> unknown package — vet manually


## 5. AI Bill of Materials (AI-SBOM)

You cannot secure a supply chain you cannot enumerate. An AI-BOM records every component — base model,
datasets, adapters, dependencies — with provenance and hash, so a later CVE or a poisoned source can
be traced to everything it touched.

In [6]:
ai_bom = {
    "system": "sovereign-rag-agent",
    "components": [
        {"type":"base_model","name":"qwen2.5:7b","source":"ollama","sha256":"a1b2...","vetted":True},
        {"type":"embedder","name":"bge-m3","source":"ollama","sha256":"c3d4...","vetted":True},
        {"type":"dataset","name":"policy-docs-v3","source":"internal","sha256":"e5f6...","vetted":True},
        {"type":"dependency","name":"langchain","version":"0.3.7","source":"pypi","vetted":False},
    ],
}
unvetted = [c["name"] for c in ai_bom["components"] if not c["vetted"]]
print(json.dumps(ai_bom, indent=1)[:300], "...")
print("\nunvetted components (block deployment until cleared):", unvetted)
print("an AI-BOM makes 'what models/data/deps are in production?' answerable in one query")

{
 "system": "sovereign-rag-agent",
 "components": [
  {
   "type": "base_model",
   "name": "qwen2.5:7b",
   "source": "ollama",
   "sha256": "a1b2...",
   "vetted": true
  },
  {
   "type": "embedder",
   "name": "bge-m3",
   "source": "ollama",
   "sha256": "c3d4...",
   "vetted": true
  },
  {
  ...

unvetted components (block deployment until cleared): ['langchain']
an AI-BOM makes 'what models/data/deps are in production?' answerable in one query


---
### What this module showed

Trust enters through the supply chain before the model runs: verify artifact **hashes** before loading
(1), never **unpickle** untrusted checkpoints — prefer safetensors (2), **pin** exact versions + hashes
and host internal packages privately (3), screen install names for **typosquats** (4), and maintain an
**AI-BOM** so every component is enumerable and traceable (5). OWASP **LLM03** — the attacks that land
inside your boundary before a prompt is ever sent.